# The sampling grid

The discrete Hankel transform does not sample a radial profile uniformly: each
harmonic order has its own radial sample positions, tied to the zeros of a
Bessel function of that order [YaoBaddour2020, Eqs. 14-15]. `pypft.PolarGrid`
is the value object describing this grid for a given `(n_radial, n_angular, R)`
-- the transform's *actual* sampling grid, as opposed to
`cartesian_to_polar`'s uniform one from the previous notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import pypft

## Order-dependent, non-uniform

Every angular row of a `PolarGrid` carries its own radial sample positions:
`grid.r[i, :]` uses Bessel order `abs(grid.harmonics[i])`, so no two rows (other
than a harmonic and its negative) share the same radial spacing. `grid.theta`
is shared by both domains and is uniform -- only the radial axis is not.

In [ ]:
grid = pypft.PolarGrid(n_radial=15, n_angular=15, R=1.0)
grid.r.shape, grid.theta.shape

A polar scatter plot of `(grid.theta, grid.r)` makes the non-uniformity, and
the resulting gap at the center, easy to see -- reproducing the shape of
[YaoBaddour2020]'s own Figs. 1-4. The sparse grid on the left matches that
paper's `R=1, N1=16, N2=15` illustration; the denser grid on the right matches
its `N1=96, N2=95` one:

In [ ]:
grid_sparse = pypft.PolarGrid(n_radial=15, n_angular=15, R=1.0)
grid_dense = pypft.PolarGrid(n_radial=95, n_angular=95, R=1.0)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), subplot_kw={"projection": "polar"})
for ax, grid_i, title in (
    (axes[0], grid_sparse, "N1=16, N2=15"),
    (axes[1], grid_dense, "N1=96, N2=95"),
):
    theta_grid = np.broadcast_to(grid_i.theta[:, np.newaxis], grid_i.r.shape)
    ax.scatter(theta_grid.ravel(), grid_i.r.ravel(), s=4)
    ax.set_title(title)
fig.tight_layout()
plt.show()

## The central gap

Both grids above leave a visible hole around the origin -- a direct
consequence of which Bessel zeros Eqs. (14)-(15) place there
[YaoBaddour2020]. Growing `n_radial` and `n_angular` shrinks the gap, but does
not close it:

In [ ]:
for n in (15, 31, 63, 127):
    grid_n = pypft.PolarGrid(n_radial=n, n_angular=n, R=1.0)
    print(f"n_radial=n_angular={n:>4}: minimum r = {grid_n.r.min():.4f}")

## The angular axis: harmonics and the unpaired Nyquist bin

The angular axis is the uniform one, so it is governed by ordinary Nyquist
sampling [YaoBaddour2020, Eq. 18] rather than by Bessel zeros: `n_angular`
samples around the circle carry exactly `n_angular` harmonics, listed by
`grid.harmonics` and centered on zero. An odd `n_angular` gives every harmonic
`+n` a partner `-n`. An even one cannot -- an even count of indices has no
symmetric layout around zero -- so the most negative harmonic,
`-n_angular // 2`, is left without a `+n_angular // 2` partner. That entry is
the angular analogue of the classical DFT's Nyquist bin, and `grid.parity`
reports which case a grid is in:

In [ ]:
for n_angular in (15, 16):
    grid_n = pypft.PolarGrid(n_radial=15, n_angular=n_angular, R=1.0)
    harmonic_list = [int(n) for n in grid_n.harmonics]
    unpaired = [n for n in harmonic_list if -n not in harmonic_list]
    print(f"n_angular={n_angular} ({grid_n.parity.name.lower()}): {harmonic_list}")
    print(f"    unpaired: {unpaired}")

An even `n_angular` is perfectly valid -- `pypft` accepts either parity and
never warns about it -- but the unpaired harmonic has two consequences worth
knowing.

- **In the grid.** The discrete Hankel transform's kernel depends only on
  `abs(n)`, so each radial row is shared by a harmonic and its negative: a
  harmonic with no partner is a row with no twin. Harmonic 0 is always its own
  partner and always untwinned, and an even `n_angular` adds a second
  untwinned row.
- **In the spectrum.** A real-valued signal's angular spectrum is conjugate
  symmetric between harmonics `+n` and `-n`. At the unpaired harmonic there is
  no partner for that symmetry to relate to, so it is one-sided there --
  `tests/dft/test_parity.py` is where this package pins that down.

The first is visible in the radial grid directly:

In [ ]:
for n_angular in (15, 16):
    grid_n = pypft.PolarGrid(n_radial=15, n_angular=n_angular, R=1.0)
    rows = [tuple(row) for row in grid_n.r]
    untwinned = [
        int(n) for row, n in zip(rows, grid_n.harmonics) if rows.count(row) == 1
    ]
    print(f"n_angular={n_angular}: radial rows with no twin, by harmonic: {untwinned}")

## Angular vs. radial resolution: a trade-off

Baddour & Chouinard's Nyquist relationship for the discrete Hankel transform
[YaoBaddour2020, Eqs. 19-21] sets a floor on how many radial samples
(`n_radial`) a given band limit needs; composed with the angular DFT, raising
`n_angular` alone (without growing `n_radial` to match) measurably degrades
accuracy instead of improving it. `pypft.check_adequacy` warns -- it never
raises -- when a grid's `n_radial` looks too small for its `n_angular`, based
on this package's own measurements of that trade-off:

In [ ]:
import warnings

for n_angular in (15, 32, 64):
    grid_n = pypft.PolarGrid(n_radial=383, n_angular=n_angular, R=40.0)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pypft.check_adequacy(grid_n)
    verdict = str(caught[0].message) if caught else "adequate, no warning"
    print(f"n_angular={n_angular:>3}, n_radial=383: {verdict}")

`n_angular` growing while `n_radial` stays fixed eventually crosses from
"adequate" to "warns" -- exactly the failure mode a fixed radial resolution
runs into as angular (spoke) count grows, e.g. for real radial MRI
acquisitions.

## Checking the Nyquist condition directly

`pypft.check_nyquist_adequacy` checks the underlying Bessel-zero condition
itself [YaoBaddour2020, Eq. 21]: the smallest zero the transform uses,
`j_(0, N1)`, must be at least `band_limit * R`. It only supports
space-limited grids and warns, rather than raising, when that condition is
violated:

In [ ]:
grid = pypft.PolarGrid(n_radial=383, n_angular=15, R=40.0)

for band_limit in (30.0, 1000.0):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        pypft.check_nyquist_adequacy(grid, band_limit)
    verdict = str(caught[0].message) if caught else "adequate, no warning"
    print(f"band_limit={band_limit:>7}: {verdict}")

### Why the order-0 zero is the binding one

Eq. 21 is a statement about order 0, but the condition behind it is not
[YaoBaddour2020, Eq. 20]. The radial grid runs one Hankel transform per
harmonic, so what has to clear `band_limit * R` is the *smallest* zero across
every Bessel order the transform uses, `min_p j_(p, N1)`. Two standard Bessel
facts collapse that minimum onto a single zero: negative orders add nothing,
since `J_(-n)` and `J_n` differ only by a sign and so share their zeros
(`j_(-n, N) = j_(n, N)`); and at a fixed index the zeros increase with the
order, `j_(0, N) < j_(1, N) < ... < j_(M, N)`. The smallest is therefore always
the order-0 one, which is why `check_nyquist_adequacy` evaluates `jn_zeros` at
order 0 alone. (`N1` is `n_radial + 1` in this package's naming, so the zero it
needs is the `(n_radial + 1)`-th.)

In [ ]:
from scipy.special import jn_zeros

orders = sorted({abs(int(n)) for n in grid.harmonics})
zeros = [float(jn_zeros(n=order, nt=grid.n_radial + 1)[-1]) for order in orders]
for order, zero in zip(orders, zeros):
    print(f"order {order}: j_({order},{grid.n_radial + 1}) = {zero:.3f}")
print(f"minimum over every order used: {min(zeros):.3f}")

## Where to go next

`PolarGrid` is the grid the forward/inverse PFT pipeline samples on; the next
notebook composes it with the angular DFT and the discrete Hankel transform to
reconstruct a signal end to end.

In [ ]:
from IPython.display import Markdown, display

from pypft.references import Reference, bibliography

display(Markdown(bibliography(Reference.YAO_BADDOUR_2020_PFT_PART2)))